In [1]:
import os, sys, numpy as np, pandas as pd, warnings

warnings.filterwarnings("ignore")
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, resnet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import matplotlib.pyplot as plt


# -------------------------------------------------
# Helper to locate a file/folder in possible input dirs
def find_path(possible_paths):
    for p in possible_paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"None of the paths exist: {possible_paths}")


# -------------------------------------------------
# Define root directories (adjusted for typical Kaggle layout)
ROOTS = [
    "../input/aerial-cactus-identification",
    "../../input/aerial-cactus-identification",
    "/kaggle/input/aerial-cactus-identification",
    "input/aerial-cactus-identification",
    "aerial-cactus-identification",
]
BASE_DIR = find_path(ROOTS)

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")
TRAIN_CSV = os.path.join(BASE_DIR, "train.csv")
SAMPLE_SUBMISSION = os.path.join(BASE_DIR, "sample_submission.csv")
print("Using base dir:", BASE_DIR)



2026-05-02 09:25:18.115261: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777713918.129246      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777713918.133608      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 09:25:18.152453: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Using base dir: ../input/aerial-cactus-identification


In [2]:
# Load training labels
train_df = pd.read_csv(TRAIN_CSV)
train_df["filepath"] = train_df["id"].apply(lambda x: os.path.join(TRAIN_DIR, x))
print("Train samples:", len(train_df))



Train samples: 14175


In [3]:
# Image data generators
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=resnet50.preprocess_input,
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.2,
    zoom_range=0.2,
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="has_cactus",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    subset="training",
    shuffle=True,
    seed=42,
)

valid_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="has_cactus",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    subset="validation",
    shuffle=False,
    seed=42,
)



Found 11340 validated image filenames.
Found 2835 validated image filenames.


In [4]:
# Build model (ResNet50 backbone)
base_model = ResNet50(
    weights="imagenet", include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False  # freeze backbone initially

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.5)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)
model.summary()



2026-05-02 09:25:27.350075: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777713927.350535      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:85:00.0, compute capability: 8.6


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 2,049 (8.00 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [5]:
# Compile and callbacks
model.compile(
    loss="binary_crossentropy",
    optimizer=Adam(learning_rate=1e-4),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, verbose=1, min_lr=1e-6
)
early_stop = EarlyStopping(
    monitor="val_loss", patience=10, verbose=1, restore_best_weights=True
)



In [6]:
# Train
EPOCHS = 5
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=valid_generator,
    callbacks=[reduce_lr, early_stop],
    verbose=2,
)

# Optional: unfreeze some layers and fine‑tune (skip to keep runtime low)


Epoch 1/5


I0000 00:00:1777713934.204248     107 service.cc:148] XLA service 0x7f6668003a80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777713934.204303     107 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-02 09:25:34.308105: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777713935.057015     107 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1777713937.152098     107 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


355/355 - 93s - 263ms/step - accuracy: 0.7722 - auc: 0.7857 - loss: 0.4931 - val_accuracy: 0.9076 - val_auc: 0.9709 - val_loss: 0.2210 - learning_rate: 1.0000e-04
Epoch 2/5
355/355 - 78s - 220ms/step - accuracy: 0.8847 - auc: 0.9428 - loss: 0.2609 - val_accuracy: 0.9397 - val_auc: 0.9833 - val_loss: 0.1604 - learning_rate: 1.0000e-04
Epoch 3/5
355/355 - 82s - 231ms/step - accuracy: 0.9194 - auc: 0.9709 - loss: 0.1905 - val_accuracy: 0.9531 - val_auc: 0.9902 - val_loss: 0.1301 - learning_rate: 1.0000e-04
Epoch 4/5
355/355 - 82s - 232ms/step - accuracy: 0.9337 - auc: 0.9801 - loss: 0.1584 - val_accuracy: 0.9538 - val_auc: 0.9921 - val_loss: 0.1159 - learning_rate: 1.0000e-04
Epoch 5/5
355/355 - 81s - 229ms/step - accuracy: 0.9456 - auc: 0.9844 - loss: 0.1408 - val_accuracy: 0.9704 - val_auc: 0.9952 - val_loss: 0.0950 - learning_rate: 1.0000e-04
Restoring model weights from the end of the best epoch: 5.


In [7]:
# Prepare test generator
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(".jpg")])
test_df = pd.DataFrame({"id": test_files})
test_df["filepath"] = test_df["id"].apply(lambda x: os.path.join(TEST_DIR, x))

test_datagen = ImageDataGenerator(preprocessing_function=resnet50.preprocess_input)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col=None,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False,
)

# Predict probabilities
preds = model.predict(test_generator, verbose=0).ravel()
test_df["has_cactus"] = preds



Found 3325 validated image filenames.


In [8]:
# Create submission file
submission_path = "submission.csv"
submission = test_df[["id", "has_cactus"]]
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")
print(submission.head())

Submission saved to submission.csv
                                     id  has_cactus
0  002930423b9840e67e5a54afd4768a1e.jpg    0.998892
1  00351838ebf6dff6e53056e00a1e307c.jpg    0.996424
2  003ec9bcef67171ba49fe4c3b7c80aec.jpg    0.999937
3  0051207eb794887c619341090de84b50.jpg    0.096018
4  0057728c8522c4881af60c3105b6492e.jpg    0.013781
